# Is daily air quality forecastable from this dataset — and for how long?

A backtest of a simple daily forecast on the XKDR measurements export, for
seven Indian cities.

**What it does.** Fits a seasonal climatology per monitoring station, carries
the latest departure from that normal forward with a weight chosen on training
years only, and scores the result against held-out 2024. Compared throughout
against two baselines a forecast has to beat to earn its place: persistence
(tomorrow equals today) and climatology (tomorrow equals the seasonal average).

**Fitted and scored PER STATION, then averaged.** Scoring a city average lets
errors at different stations cancel inside the metric — by the triangle
inequality `|mean(ŷ) − mean(y)| ≤ mean(|ŷᵢ − yᵢ|)`, so the city figure can
never be the larger of the two. Measured here it is roughly 71–82% of the
per-station error, which flatters the model by about a quarter.

**The finding that motivated sharing this.** Forecast skill collapses with the
age of the most recent reading. With same-day data the model beats climatology
by around a third in Delhi; one day old, a fifth; by four days old there is
almost nothing left. Publication latency is not a convenience issue for anyone
trying to forecast from this data — it is the binding constraint.

---

**To run:** `pip install -r requirements.txt`, then open this notebook from the
directory containing `baselines.py`. It reads the parquet export directly via
duckdb — no database, no API key. Set `XKDR_DATA_DIR` if the measurements are
not in a `data/` directory beside the notebook.

Roughly two minutes end to end, most of it the first duckdb scan. Loaders cache
their aggregates beside the data as parquet, so later runs are seconds.

## Setup

In [ ]:
# This notebook imports baselines.py from the same directory.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
# baselines.py sits beside this notebook. Set XKDR_DATA_DIR if the measurements
# parquet is not in a `data/` directory beside it too.
sys.path.insert(0, str(Path.cwd()))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from baselines import (
    load_city_daily, city_series, climatology, backtest, predictions_for,
    naqi_band, naqi_band_aqi, load_city_daily_aqi, aqi_series, NAQI_LABELS, SEASON_MONTHS, CITIES,
    # Per-station equivalents of backtest/predictions_for. Same shapes out,
    # so the tables and figures below are unchanged -- only what they are
    # measuring changes.
    load_station_daily, load_station_daily_aqi,
    backtest_stations, predictions_for_stations)
HORIZONS = [1, 2, 3, 4, 5, 6, 7] # Number of days ahead to forecast

# Model colours, fixed so every figure reads the same way.
M_COLOR = {'persistence': '#eb6834', 'climatology': '#1baf7a', 'blend': '#2a78d6'}
# One colour per city, in CITIES order. Assigned once and never cycled, so a
# city keeps its colour across every figure in the notebook.
CITY_COLOR = dict(zip(CITIES, ['#2a78d6', '#eb6834', '#1baf7a',
                               '#eda100', '#4a3aa7', '#e34948',
                               "#EC0CE1"]))
M_LABEL = {'persistence': 'Persistence', 'climatology': 'Climatology', 'blend': 'Blend (our model)'}
INK, MUTED, GRID = '#12171b', '#5c6b73', '#e8e8e6'
pd.set_option('display.width', 130)

# Every per-city figure sizes its grid from len(CITIES) rather than a literal.
def city_grid(ncol=3, w=4.7, h=4.3, **kw):
    """Axes grid sized to CITIES; returns (fig, flat_axes) with spares hidden."""
    nrow = -(-len(CITIES) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(w * ncol, h * nrow), **kw)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(CITIES):]:
        ax.axis('off')
    return fig, axes

## Load data (PM2.5 and composite AQI)

In [ ]:
# Load historical daily data for each city and print a summary of the number of days of data available for each city.
daily_pm25 = load_city_daily(parameter="PM2.5")
series_pm25 = {c: city_series(daily_pm25, c) for c in CITIES}
print('Days of daily PM2.5 data available for each city:')
for c, s in series_pm25.items():
    print(f'{c:10} {int(s.notna().sum()):>5} days   {s.index.min().date()} to {s.index.max().date()}')


daily_aqi = load_city_daily_aqi()   # builds data/xkdr_city_daily_aqi.parquet on first call
series_aqi = {c: aqi_series(daily_aqi, c) for c in CITIES}
print()
print('Days of daily AQI data available for each city:')
for c, s in series_aqi.items():
    print(f'{c:10} {int(s.notna().sum()):>5} days   {s.index.min().date()} to {s.index.max().date()}')

print()
print('WHICH POLLUTANT ACTUALLY DRIVES THE AQI? (% of days)')
share = (daily_aqi.dominant.value_counts(normalize=True) * 100).round(1)
print(share.to_string())
print()
by_city = (pd.crosstab(daily_aqi.city, daily_aqi.dominant, normalize='index') * 100)
print(by_city.reindex(CITIES).round(1).to_string())


# ─── Station-level data: what this notebook actually fits on ─────────────────
# The city series above are kept because the markdown refers to them and the
# coverage check below uses them to pick a test year. Everything that TRAINS a
# model now uses these instead.
#
# Why: the app never serves a forecast trained on a 70-station mean. It serves
# a place -- a few stations around a point the user chose. Scoring the city
# average also lets errors at different stations cancel inside the metric, so
# it reports a number nobody experiences. See the note above
# backtest_stations() in baselines.py for the inequality that guarantees it.
station_pm25 = load_station_daily()
station_aqi = load_station_daily_aqi()

print()
print('STATIONS WITH ENOUGH HISTORY TO FIT (>=365 days before the test year)')
for c in CITIES:
    n_pm = station_pm25[station_pm25.city == c].station.nunique()
    n_aq = station_aqi[station_aqi.city == c].station.nunique()
    print(f'{c:10} PM2.5 {n_pm:>3} stations   AQI {n_aq:>3} stations')


## Choose the test year

**2024 is the most recent year with data for the pollution season**, so that's the test year. 2019–2023
are used as repeat trials further down to check the answer isn't a fluke of one year.

In [ ]:
coverage = []
for c, s in series_pm25.items():
    for y in range(2018, 2026):
        yr = s[s.index.year == y]
        season = yr[yr.index.month.isin(SEASON_MONTHS)]
        coverage.append({'city': c, 'year': y, 'days': int(yr.notna().sum()),
                         'season_days': int(season.notna().sum())})
cov = pd.DataFrame(coverage).pivot(index='year', columns='city', values='season_days')
print('DAYS OF IN-SEASON DATA PER YEAR for PM2.5 (SEASON MONTHS: {}-{})'.format(SEASON_MONTHS[0], SEASON_MONTHS[-1]))
print(cov.to_string())
print()

coverage = []
for c, s in series_aqi.items():
    for y in range(2018, 2026):
        yr = s[s.index.year == y]
        season = yr[yr.index.month.isin(SEASON_MONTHS)]
        coverage.append({'city': c, 'year': y, 'days': int(yr.notna().sum()),
                         'season_days': int(season.notna().sum())})
cov = pd.DataFrame(coverage).pivot(index='year', columns='city', values='season_days')
print('DAYS OF IN-SEASON DATA PER YEAR for AQI (SEASON MONTHS: {}-{})'.format(SEASON_MONTHS[0], SEASON_MONTHS[-1]))
print(cov.to_string())
print()

print('Chosen test year: 2024')

TEST_YEAR = 2024

## Train the model and predict daily average values for the test year

In [ ]:
# Fitted and scored PER STATION, then averaged across the stations of each city.
#
# Each station gets its own climatology and its own alpha, because that is how
# both are served. The MAE reported is the unweighted mean of the stations'
# MAEs, so a monitor that reports twice as often does not get twice the say.
#
# This is the one change that moves every number below. On composite AQI,
# Delhi day+1 goes from 40.4 (city average) to 49.3 (per station) -- the city
# figure is about 82% of the honest one, because averaging before taking the
# absolute value lets errors at different stations cancel.
results_pm25 = pd.concat([backtest_stations(station_pm25, c, TEST_YEAR, HORIZONS) for c in CITIES],
                    ignore_index=True)
results_aqi = pd.concat([backtest_stations(station_aqi, c, TEST_YEAR, HORIZONS) for c in CITIES],
                    ignore_index=True)

print(f'stations behind each row: '
      f"PM2.5 {results_pm25.n_stations.min()}-{results_pm25.n_stations.max()}, "
      f"AQI {results_aqi.n_stations.min()}-{results_aqi.n_stations.max()}")

---
## Quality check 1: Mean absolute error in predicting Oct - Jan 2024 daily average

### PM2.5

In [ ]:
season_pm25 = results_pm25[results_pm25.subset == 'season']
tbl = (season_pm25.pivot_table(index=['city', 'horizon'], columns='model', values='mae')
             .reindex(columns=['persistence', 'climatology', 'blend']).round(1))
tbl['best'] = tbl.idxmin(axis=1)
print(f'SEASON (Oct-Jan) MAE of PM2.5 in ug/m3, test year {TEST_YEAR}')
print(tbl.to_string())

In [ ]:
fig, axes = city_grid(ncol=3, w=4.7, h=4.3, sharex=True)

for ax, city in zip(axes, CITIES):
    d = season_pm25[season_pm25.city == city]
    for m in ['persistence', 'climatology', 'blend']:
        dd = d[d.model == m].sort_values('horizon')
        ax.plot(dd.horizon, dd.mae, marker='o', markersize=5, linewidth=2.2,
                color=M_COLOR[m], label=M_LABEL[m], zorder=3)

    # The crossover: first horizon where the blend no longer beats climatology.
    b = d[d.model == 'blend'].set_index('horizon').mae
    cl = d[d.model == 'climatology'].set_index('horizon').mae
    lost = [h for h in sorted(b.index) if b[h] >= cl[h]]
    if lost:
        h0 = lost[0]
        ax.axvspan(h0 - 0.5, max(HORIZONS) + 0.5, color='#b3261e', alpha=0.07, zorder=0)
        ax.text(h0 + 0.1, ax.get_ylim()[1], 'no better than\nthe seasonal average',
                fontsize=8.5, color='#b3261e', va='top', ha='left')

    mean_level = series_pm25[city][series_pm25[city].index.year == TEST_YEAR]
    mean_level = mean_level[mean_level.index.month.isin(SEASON_MONTHS)].mean()
    ax.set_title(f'{city}\nseason mean {mean_level:.0f} µg/m³', loc='left',
                 fontsize=11.5, fontweight='bold', pad=8)
    ax.set_xlabel('days ahead')
    ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    ax.set_xticks(HORIZONS)
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')

for i in range(0, len(axes), 3):
    axes[i].set_ylabel('mean absolute error, PM2.5 (µg/m³)')
    axes[i].legend(frameon=False, fontsize=9.5, loc='lower right')
fig.suptitle('How far ahead can we forecast PM2.5?  Pollution season, test year 2024',
             fontsize=13, fontweight='bold', x=0.06, ha='left', y=0.99)
fig.text(0.06, 0.935, 'Lower is better. Where the blue line meets the green one, our model '
         'has collapsed into the seasonal average and stops being a forecast.',
         fontsize=9.5, color=MUTED, ha='left')
fig.tight_layout(rect=[0, 0, 1, 0.87]); plt.show()

### Composite AQI

In [ ]:
season_aqi = results_aqi[results_aqi.subset == 'season']
tbl = (season_aqi.pivot_table(index=['city', 'horizon'], columns='model', values='mae')
             .reindex(columns=['persistence', 'climatology', 'blend']).round(1))
tbl['best'] = tbl.idxmin(axis=1)
print(f'SEASON (Oct-Jan) MAE of AQI, test year {TEST_YEAR}')
print(tbl.to_string())

In [ ]:
fig, axes = city_grid(ncol=3, w=4.7, h=4.3, sharex=True)

for ax, city in zip(axes, CITIES):
    d = season_aqi[season_aqi.city == city]
    for m in ['persistence', 'climatology', 'blend']:
        dd = d[d.model == m].sort_values('horizon')
        ax.plot(dd.horizon, dd.mae, marker='o', markersize=5, linewidth=2.2,
                color=M_COLOR[m], label=M_LABEL[m], zorder=3)

    # The crossover: first horizon where the blend no longer beats climatology.
    b = d[d.model == 'blend'].set_index('horizon').mae
    cl = d[d.model == 'climatology'].set_index('horizon').mae
    lost = [h for h in sorted(b.index) if b[h] >= cl[h]]
    if lost:
        h0 = lost[0]
        ax.axvspan(h0 - 0.5, max(HORIZONS) + 0.5, color='#b3261e', alpha=0.07, zorder=0)
        ax.text(h0 + 0.1, ax.get_ylim()[1], 'no better than\nthe seasonal average',
                fontsize=8.5, color='#b3261e', va='top', ha='left')

    mean_level = series_aqi[city][series_aqi[city].index.year == TEST_YEAR]
    mean_level = mean_level[mean_level.index.month.isin(SEASON_MONTHS)].mean()
    ax.set_title(f'{city}\nseason mean {mean_level:.0f}', loc='left',
                 fontsize=11.5, fontweight='bold', pad=8)
    ax.set_xlabel('days ahead')
    ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    ax.set_xticks(HORIZONS)
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')

for i in range(0, len(axes), 3):
    axes[i].set_ylabel('mean absolute error, AQI')
    axes[i].legend(frameon=False, fontsize=9.5, loc='lower right')
fig.suptitle('How far ahead can we forecast AQI?  Pollution season, test year 2024',
             fontsize=13, fontweight='bold', x=0.06, ha='left', y=0.99)
fig.text(0.06, 0.935, 'Lower is better. Where the blue line meets the green one, our model '
         'has collapsed into the seasonal average and stops being a forecast.',
         fontsize=9.5, color=MUTED, ha='left')
fig.tight_layout(rect=[0, 0, 1, 0.87]); plt.show()

## Quality check 2: percent improvement over the better of the two baselines (climatology/persistence)

### PM2.5

In [ ]:
skill = []
for city in CITIES:
    d = season_pm25[season_pm25.city == city]
    for h in HORIZONS:
        dh = d[d.horizon == h].set_index('model').mae
        base = min(dh['persistence'], dh['climatology'])
        skill.append({'city': city, 'horizon': h,
                      'skill_pct': 100 * (base - dh['blend']) / base})
skill = pd.DataFrame(skill)

fig, ax = plt.subplots(figsize=(10, 4.4))
n_c = len(CITIES)
w = 0.8 / n_c
for i, city in enumerate(CITIES):
    d = skill[skill.city == city].sort_values('horizon')
    offset = (i - (n_c - 1) / 2) * w
    ax.bar(d.horizon + offset, d.skill_pct, width=w,
           color=CITY_COLOR[city], label=city, zorder=3)
ax.axhline(0, color=INK, linewidth=1.2, zorder=4)
ax.axhspan(-30, 0, color='#b3261e', alpha=0.06, zorder=0)
ax.set_xlabel('days ahead'); ax.set_ylabel('% better than the best baseline')
ax.set_xticks(HORIZONS)
ax.set_title('PM2.5 forecast skill decays fast', loc='left', fontsize=12.5, fontweight='bold', pad=26)
ax.text(0, 1.06, 'Above zero the model earns its place. At or below zero, we should use the seasonal average or the persistence model.', transform=ax.transAxes,
        fontsize=9.5, color=MUTED)
ax.legend(frameon=False, fontsize=9, ncol=6)
ax.grid(axis='y', color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')
plt.tight_layout(); plt.show()
print(skill.pivot(index='horizon', columns='city', values='skill_pct').round(1).to_string())

### AQI

In [ ]:
skill = []
for city in CITIES:
    d = season_aqi[season_aqi.city == city]
    for h in HORIZONS:
        dh = d[d.horizon == h].set_index('model').mae
        base = min(dh['persistence'], dh['climatology'])
        skill.append({'city': city, 'horizon': h,
                      'skill_pct': 100 * (base - dh['blend']) / base})
skill = pd.DataFrame(skill)

fig, ax = plt.subplots(figsize=(10, 4.4))
n_c = len(CITIES)
w = 0.8 / n_c
for i, city in enumerate(CITIES):
    d = skill[skill.city == city].sort_values('horizon')
    offset = (i - (n_c - 1) / 2) * w
    ax.bar(d.horizon + offset, d.skill_pct, width=w,
           color=CITY_COLOR[city], label=city, zorder=3)
ax.axhline(0, color=INK, linewidth=1.2, zorder=4)
ax.axhspan(-30, 0, color='#b3261e', alpha=0.06, zorder=0)
ax.set_xlabel('days ahead'); ax.set_ylabel('% better than the best baseline')
ax.set_xticks(HORIZONS)
ax.set_title('AQI forecast skill decays fast', loc='left', fontsize=12.5, fontweight='bold', pad=26)
ax.text(0, 1.06, 'Above zero the model earns its place. At or below zero, we should use the seasonal average or the persistence model.', transform=ax.transAxes,
        fontsize=9.5, color=MUTED)
ax.legend(frameon=False, fontsize=9, ncol=6)
ax.grid(axis='y', color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')
plt.tight_layout(); plt.show()
print(skill.pivot(index='horizon', columns='city', values='skill_pct').round(1).to_string())

### AQI vs PM2.5 forecasting skill at +1 day

In [ ]:
# Per station, then averaged -- same change as cell 8. The level column is
# still read off the city series, because it is a descriptive "how dirty is
# this city" figure rather than anything the model is scored against.
aqi_rows = []
for c in CITIES:
    for label, frame, cser in (('AQI', station_aqi, series_aqi[c]),
                               ('PM2.5', station_pm25, series_pm25[c])):
        r = backtest_stations(frame, c, TEST_YEAR, [1])
        if r.empty: continue
        se = r[(r.subset == 'season') & (r.horizon == 1)]
        if se.empty or se.n.max() < 40: continue
        dh = se.set_index('model').mae
        te = cser[cser.index.year == TEST_YEAR]
        lvl = te[te.index.month.isin(SEASON_MONTHS)].mean()
        base = min(dh['persistence'], dh['climatology'])
        aqi_rows.append({'city': c, 'target': label, 'level': lvl, 'mae': dh.min(),
                         'best_model': dh.idxmin(),
                         'skill': 100 * (base - dh.min()) / base if base != 0 else 0})
aqi_cmp = pd.DataFrame(aqi_rows)

piv = aqi_cmp.pivot(index='city', columns='target', values='skill').reindex(CITIES)
piv['difference'] = (piv['AQI'] - piv['PM2.5']).round(1)
print('SKILL AT +1 DAY — composite AQI vs PM2.5 (% better than baseline (climatology or persistence))')
print(piv.round(1).to_string())
print()
print(f"  mean: AQI {aqi_cmp[aqi_cmp.target=='AQI'].skill.mean():.1f}%   "
      f"PM2.5 {aqi_cmp[aqi_cmp.target=='PM2.5'].skill.mean():.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
x = np.arange(len(piv)); w = 0.38
ax.bar(x - w/2, piv['PM2.5'], width=w, color='#2a78d6', label='PM2.5', zorder=3)
ax.bar(x + w/2, piv['AQI'], width=w, color='#eb6834', label='composite AQI', zorder=3)
ax.set_xticks(x); ax.set_xticklabels(piv.index, rotation=20, ha='right')
ax.set_ylabel('% better than baseline (cimatology or persistence)')
ax.set_title('AQI forecasts about as well as PM2.5', loc='left',
             fontsize=11.5, fontweight='bold', pad=8)
ax.legend(frameon=False, fontsize=9)

ax = axes[1]
bc = by_city.reindex(CITIES)
bottom = np.zeros(len(bc))
for pol, col in (('pm10', '#eb6834'), ('pm25', '#2a78d6'),
                 ('no2', '#1baf7a'), ('so2', '#eda100')):
    if pol not in bc: continue
    ax.bar(bc.index, bc[pol], bottom=bottom, color=col, label=pol, zorder=3)
    bottom += bc[pol].values
ax.set_ylabel('% of days driving the AQI')
ax.set_xticklabels(bc.index, rotation=20, ha='right')
ax.set_title('…but PM10 is what it is usually tracking', loc='left',
             fontsize=11.5, fontweight='bold', pad=8)
ax.legend(frameon=False, fontsize=9, ncol=4)

for a in axes:
    a.grid(axis='y', color=GRID, linewidth=0.8); a.set_axisbelow(True)
    for sp in ('top', 'right'): a.spines[sp].set_visible(False)
    a.spines['left'].set_color('0.7'); a.spines['bottom'].set_color('0.7')
plt.tight_layout(); plt.show()

## Quality check 4: How often do we predict the correct AQI/PM2.5 band?

People act based on pollution level category, not the absolute number. So the product-relevant score is how often the forecast lands in the
correct CPCB band, or within one band of it.

### PM2.5

In [ ]:
# Band agreement is computed PER STATION and then averaged, not on a city
# average. Banding is a threshold, so cancellation between stations flatters it
# even more than it flatters MAE -- on composite AQI, Mumbai reads 85% band
# agreement city-wide and 72% per station.
band_rows = []
for city in CITIES:
    for h in HORIZONS:
        ps = predictions_for_stations(station_pm25, city, TEST_YEAR, h)
        if ps.empty: continue
        ps = ps[ps.index.month.isin(SEASON_MONTHS)].dropna(subset=['actual', 'blend'])
        exact, within1, n_total = [], [], 0
        for station, p in ps.groupby('station'):
            if len(p) < 30: continue
            a = naqi_band(p.actual.values).codes
            f = naqi_band(p.blend.values).codes
            ok = a >= 0
            if ok.sum() == 0: continue
            exact.append(100 * float((a[ok] == f[ok]).mean()))
            within1.append(100 * float((np.abs(a[ok] - f[ok]) <= 1).mean()))
            n_total += int(ok.sum())
        if not exact: continue
        band_rows.append({'city': city, 'horizon': h, 'n': n_total,
                          'n_stations': len(exact),
                          'exact_pct': float(np.mean(exact)),
                          'within1_pct': float(np.mean(within1))})
bands = pd.DataFrame(band_rows)
print('FORECAST LANDS IN THE RIGHT NAQI BAND (%), pollution season')
print(bands.pivot(index='horizon', columns='city', values='exact_pct').round(0).to_string())

fig, ax = plt.subplots(figsize=(10, 4.4))
for city in CITIES:
    d = bands[bands.city == city].sort_values('horizon')
    if d.empty: continue
    col = CITY_COLOR[city]
    ax.plot(d.horizon, d.exact_pct, marker='o', linewidth=2.2, color=col, label=f'{city} — exact')
    ax.plot(d.horizon, d.within1_pct, marker='o', markersize=4, linewidth=1.3,
            linestyle=':', color=col, alpha=0.85)
ax.set_xlabel('days ahead'); ax.set_ylabel('% of days')
ax.set_xticks(HORIZONS); ax.set_ylim(0, 105)
ax.set_title('How often the PM2.5 forecast gives the right advice', loc='left',
             fontsize=12.5, fontweight='bold', pad=26)
ax.text(0, 1.06, 'Solid: exactly the right CPCB band. Dotted: within one band.',
        transform=ax.transAxes, fontsize=9.5, color=MUTED)
ax.legend(frameon=False, fontsize=9, ncol=3, loc='lower left')
ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')
plt.tight_layout(); plt.show()

### AQI

In [ ]:
# Band agreement is computed PER STATION and then averaged, not on a city
# average. Banding is a threshold, so cancellation between stations flatters it
# even more than it flatters MAE -- on composite AQI, Mumbai reads 85% band
# agreement city-wide and 72% per station.
band_rows = []
for city in CITIES:
    for h in HORIZONS:
        ps = predictions_for_stations(station_aqi, city, TEST_YEAR, h)
        if ps.empty: continue
        ps = ps[ps.index.month.isin(SEASON_MONTHS)].dropna(subset=['actual', 'blend'])
        exact, within1, n_total = [], [], 0
        for station, p in ps.groupby('station'):
            if len(p) < 30: continue
            a = naqi_band_aqi(p.actual.values).codes
            f = naqi_band_aqi(p.blend.values).codes
            ok = a >= 0
            if ok.sum() == 0: continue
            exact.append(100 * float((a[ok] == f[ok]).mean()))
            within1.append(100 * float((np.abs(a[ok] - f[ok]) <= 1).mean()))
            n_total += int(ok.sum())
        if not exact: continue
        band_rows.append({'city': city, 'horizon': h, 'n': n_total,
                          'n_stations': len(exact),
                          'exact_pct': float(np.mean(exact)),
                          'within1_pct': float(np.mean(within1))})
bands = pd.DataFrame(band_rows)
print('FORECAST LANDS IN THE RIGHT NAQI BAND (%), pollution season')
print(bands.pivot(index='horizon', columns='city', values='exact_pct').round(0).to_string())

fig, ax = plt.subplots(figsize=(10, 4.4))
for city in CITIES:
    d = bands[bands.city == city].sort_values('horizon')
    if d.empty: continue
    col = CITY_COLOR[city]
    ax.plot(d.horizon, d.exact_pct, marker='o', linewidth=2.2, color=col, label=f'{city} — exact')
    ax.plot(d.horizon, d.within1_pct, marker='o', markersize=4, linewidth=1.3,
            linestyle=':', color=col, alpha=0.85)
ax.set_xlabel('days ahead'); ax.set_ylabel('% of days')
ax.set_xticks(HORIZONS); ax.set_ylim(0, 105)
ax.set_title('How often the AQI forecast gives the right advice', loc='left',
             fontsize=12.5, fontweight='bold', pad=26)
ax.text(0, 1.06, 'Solid: exactly the right CPCB band. Dotted: within one band.',
        transform=ax.transAxes, fontsize=9.5, color=MUTED)
ax.legend(frameon=False, fontsize=9, ncol=3, loc='lower left')
ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')
plt.tight_layout(); plt.show()

## How stale input degrades the forecast

This is the link between the forecast and the live-data problem. The model's only current
information is the most recent observation. If that reading is 4 days old — which is what
the OpenAQ archive gives us — then a forecast for *tomorrow* is really a 5-day-ahead
forecast, and lands in the range where the model adds nothing.

In [ ]:
# `season` was renamed season_pm25 when the notebook was restructured.
# These numbers are now per-station averages, so the climatology baseline
# and the blend MAEs are both larger than the city-average versions the
# surrounding text was written against.
lag_rows = []
d = season_pm25[(season_pm25.city == 'Delhi')]
b = d[d.model == 'blend'].set_index('horizon').mae
cl = float(d[d.model == 'climatology'].mae.iloc[0])
for data_lag in [0, 1, 2, 4]:
    for want in [1, 2, 3]:
        eff = want + data_lag
        lag_rows.append({'data_age_days': data_lag, 'forecast_for': f'+{want}d',
                         'effective_horizon': eff,
                         'mae': b.get(eff, np.nan)})
lag = pd.DataFrame(lag_rows)
lag['beats_average'] = np.where(lag.mae < cl, 'yes', 'no')
print(f'DELHI season_pm25 — climatology baseline is {cl:.0f} ug/m3')
print(lag.round(1).to_string(index=False))
print()
print('With 4-day-old input (OpenAQ archive), no forecast we can offer beats the average.')

---
## How stale can the input be?

The model's only current information is the **last observation**. So if that reading is
already `L` days old, forecasting `W` days ahead is really an `L+W` step problem — the
anomaly being carried forward has had `L+W` days to decay, not `W`.

This is not academic: our three possible sources differ enormously in freshness, and the
choice between them decides whether a forecast is worth showing at all.

| source | lag | 
|---|---|
| OpenAQ live API | **~1 day** (publishes ~17 h behind) |
| OpenAQ S3 archive | **3–4 days** |
| XKDR | ~6 months — history only |

The bar stays the same throughout: beat **climatology**, which needs no current reading at
all. A forecast built on input so old that it loses to the seasonal average is not a
forecast — it is a worse way of drawing the average.

In [ ]:
from baselines import backtest_stale_input, per_station_table

# Per station, then averaged. `series` no longer exists after the
# restructure; this was raising NameError before the conversion.
stale = pd.concat([per_station_table(station_pm25, c, backtest_stale_input, TEST_YEAR)
                   for c in CITIES],
                  ignore_index=True)

for city in CITIES:
    d = stale[stale.city == city]
    if d.empty: continue
    print(f'{city}  —  season MAE (climatology = {d.clim_mae.iloc[0]:.1f})')
    print(d.pivot(index='data_lag', columns='forecast_for', values='mae').round(1).to_string())
    print()

In [ ]:
fig, axes = city_grid(ncol=3, w=4.5, h=4.0)

for ax, city in zip(axes, CITIES):
    d = stale[stale.city == city]
    if d.empty:
        ax.axis('off'); continue
    grid = d.pivot(index='data_lag', columns='forecast_for', values='skill_vs_clim')
    # Diverging around zero: above zero the forecast earns its place, below it loses
    # to an average that needs no data at all.
    lim = max(abs(grid.values).max(), 1)
    im = ax.imshow(grid.values, cmap='RdYlGn', vmin=-lim, vmax=lim, aspect='auto')
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            v = grid.values[i, j]
            ax.text(j, i, f'{v:+.0f}%', ha='center', va='center', fontsize=10,
                    color='#12171b', fontweight='bold' if v > 0 else 'normal')
    ax.set_xticks(range(grid.shape[1])); ax.set_xticklabels([f'+{c}d' for c in grid.columns])
    ax.set_yticks(range(grid.shape[0])); ax.set_yticklabels([f'{r}d old' for r in grid.index])
    ax.set_xlabel('forecast for'); 
    ax.set_title(city, loc='left', fontsize=11.5, fontweight='bold', pad=8)
    for sp in ('top','right','left','bottom'): ax.spines[sp].set_visible(False)
    ax.tick_params(length=0)
for i in range(0, len(axes), 3):
    axes[i].set_ylabel('input data age')

fig.suptitle('Forecast skill against the seasonal average, by how stale the input is',
             fontsize=13, fontweight='bold', x=0.045, ha='left', y=0.985)
fig.text(0.045, 0.925, 'Green: the forecast beats the seasonal average. Red: it loses, and '
         'we should show the average instead.', fontsize=9.5, color=MUTED, ha='left')
fig.tight_layout(rect=[0, 0, 1, 0.855]); plt.show()

In [ ]:
# The practical question: which source supports which promise?
SOURCES = {'OpenAQ live API (~1 day)': 1, 'OpenAQ S3 archive (3 days)': 3,
           'OpenAQ S3 archive (4 days)': 4}
print('DELHI — can we honestly offer this?')
d = stale[stale.city == 'Delhi']
for label, lag in SOURCES.items():
    bits = []
    for w in (1, 2, 3):
        row = d[(d.data_lag == lag) & (d.forecast_for == w)]
        if row.empty: continue
        r = row.iloc[0]
        bits.append(f"+{w}d {r.mae:5.1f} ({r.skill_vs_clim:+5.1f}%) {'yes' if r.beats_clim else 'NO '}")
    print(f'  {label:<28} ' + '   '.join(bits))
print()
print('  climatology (no current reading needed):', f"{d.clim_mae.iloc[0]:.1f}")

### What this says

**A one-day-old reading still supports a real forecast.** At `+1d` from day-old input the
error is well under climatology — roughly the quality you'd get forecasting *two* days ahead
from live data, which is still worth showing.

**A four-day-old reading does not.** By the time the input is four days old, a "tomorrow"
forecast is a five-step problem and lands within a whisker of the seasonal average; the
three-day version is actively worse than it. At that lag the honest move is to stop calling
it a forecast and show the seasonal normal, which is free and no worse.

**So the source decides the feature.** This is the strongest practical argument for keeping
the live API path working rather than falling back to the archive: it is the difference
between a forecast and a long-run average with extra steps.

---
## How much recent data does the forecast actually need?

The model carries forward the latest departure from the seasonal normal. A reasonable
instinct is that averaging that departure over several recent days would be steadier than
trusting one. This tests it — refitting α per window on training years only, so each
window is judged at its own best setting.

This is not an idle question: the answer sets how much raw data we have to keep in the
database, and therefore whether the app fits in Supabase's free tier.

In [ ]:
from baselines import backtest_anomaly_window, gap_stats, per_station_table

# Per station, then averaged -- the anomaly is carried forward per station.
win = pd.concat([per_station_table(station_pm25, c, backtest_anomaly_window, TEST_YEAR)
                 for c in CITIES],
                ignore_index=True)
tbl = win.pivot(index='city', columns='window_days', values='mae').round(1)
tbl = tbl.reindex(CITIES)
tbl['best'] = tbl.idxmin(axis=1)
print('MAE at +1 day by anomaly-averaging window (season, test '
      f'{TEST_YEAR}) — lower is better')
print(tbl.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
for city in CITIES:
    d = win[win.city == city].sort_values('window_days')
    if d.empty: continue
    # Indexed to each city's own N=1 error, so six very different pollution
    # levels can share one axis and the SHAPE is what gets compared.
    base = d[d.window_days == 1].mae.iloc[0]
    ax.plot(d.window_days, 100 * d.mae / base, marker='o', markersize=4,
            linewidth=2, color=CITY_COLOR[city], label=city)
ax.axhline(100, color=INK, linewidth=1.2)
ax.set_xscale('log'); ax.set_xticks([1, 2, 3, 5, 7, 14, 30])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('days of recent data averaged into the anomaly')
ax.set_ylabel('error, % of the 1-day result')
ax.set_title('One day of recent data beats any longer window', loc='left',
             fontsize=12.5, fontweight='bold', pad=26)
ax.text(0, 1.06, 'Every city degrades monotonically as older days are mixed in. '
        'Pollution episodes are short: older days dilute the signal.',
        transform=ax.transAxes, fontsize=9.5, color=MUTED)
ax.legend(frameon=False, fontsize=9, ncol=6)
ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for s in ('top','right'): ax.spines[s].set_visible(False)
ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')
plt.tight_layout(); plt.show()

### …but the forecast has to be able to *find* that day

Reading one day only works if one is there. This measures how far back we must look to
find any reading, which is what actually sets the raw-retention window.

One caveat handled in the code: the XKDR export is missing all of Q1 2025 for every city.
That single 90-day outage would dominate the percentiles, so runs longer than 30 days are
excluded as outages rather than normal behaviour.

In [ ]:
# Asked PER STATION and averaged. The app looks for a recent reading at a
# station, not in a city average -- and a city average is never missing,
# because some station always reported. Measuring gaps on it would say the
# data is far more continuous than any single monitor actually is.
import numpy as _np
_rows = {}
for c in CITIES:
    per = [gap_stats(s) for s in
           [g.groupby('d')['v'].mean().asfreq('D')
            for _, g in station_pm25[station_pm25.city == c].groupby('station')]]
    per = [p for p in per if p]
    if per:
        _rows[c] = {k: float(_np.mean([p[k] for p in per])) for k in per[0]}
gaps = pd.DataFrame(_rows).T
gaps = gaps.reindex(CITIES)
print('DAYS BACK TO THE MOST RECENT AVAILABLE READING (2019+, outages excluded)')
print(gaps[['p50', 'p90', 'p99', 'p999', 'missing_days', 'total_days']].to_string())
print()
print(f'  worst p99.9 across cities: {gaps.p999.max():.0f} days')
print(f'  -> a {int(np.ceil(gaps.p999.max() / 10) * 10)}-day raw window covers 99.9% of days')

---
## Forecasting a time of day, not just a day

"Should I go for a run at 3pm?" needs an hourly answer. The approach is a multiplicative
decomposition — forecast the day, then bend it by the hour:

$$\hat{y}(\text{day}, \text{hour}) = \hat{y}_{\text{daily}}(\text{day}) \times s(\text{city}, \text{month}, \text{hour})$$

where $s$ is the historical mean of *value ÷ that day's mean*. Because it is a **ratio**
it scales with the level: a shape learned in a clean month still applies in a dirty one.

First, is a shape even the right object to model? Splitting hourly variance three ways:

In [ ]:
from baselines import (load_city_hourly, variance_decomposition,
                                        diurnal_shape, backtest_hour, best_hour_eval,
                                        complete_days)

hourly = load_city_hourly()   # builds data/xkdr_city_hourly.parquet on first call
print(f'{len(hourly):,} city-hours, {hourly.d.min().date()} to {hourly.d.max().date()}')

vd = pd.DataFrame({c: variance_decomposition(hourly[hourly.city == c])
                   for c in CITIES if (hourly.city == c).any()}).T.reindex(CITIES)
print()
print('SHARE OF HOURLY VARIANCE EXPLAINED BY')
print((vd * 100).round(1).rename(columns={
    'day': 'which day it is', 'diurnal': 'hour-of-day shape',
    'residual': 'neither'}).to_string())

The daily mean dominates everywhere — 59% to 88%. The diurnal shape adds only another
3–8%.

That is the argument for this decomposition rather than twenty-four separate hourly
models: **getting the day right is most of the job**, and the shape is a real but
second-order correction. Per-hour models would spend their capacity chasing the residual,
which is weather, not time of day.

In [ ]:
hb = pd.concat([backtest_hour(hourly[hourly.city == c], TEST_YEAR) for c in CITIES
                if (hourly.city == c).any()], ignore_index=True)
piv = hb.pivot_table(index=['city', 'hour'], columns='model', values='mae')
piv = piv[['flat', 'shape', 'same_hour', 'hour_clim']].round(1)
print('MAE FORECASTING A NAMED HOUR, 1 DAY AHEAD (season, test '
      f'{TEST_YEAR})')
print(piv.to_string())
print()
means = hb.groupby('model').mae.mean()
flat = means['flat']
for m in ['flat', 'shape', 'same_hour', 'hour_clim']:
    delta = f'  ({100 * (means[m] - flat) / flat:+.0f}% vs flat)' if m != 'flat' else ''
    print(f'  {m:<11} {means[m]:5.1f}{delta}')

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.8))
order = [(c, h) for c in CITIES for h in (9, 15, 18)]
order = [x for x in order if ((hb.city == x[0]) & (hb.hour == x[1])).any()]
x = np.arange(len(order)); w = 0.38
flat_v = [hb[(hb.city == c) & (hb.hour == h) & (hb.model == 'flat')].mae.iloc[0] for c, h in order]
shape_v = [hb[(hb.city == c) & (hb.hour == h) & (hb.model == 'shape')].mae.iloc[0] for c, h in order]
ax.bar(x - w/2, flat_v, width=w, color='#b6c3c9', label='flat daily number', zorder=3)
ax.bar(x + w/2, shape_v, width=w, color='#2a78d6', label='daily × diurnal shape', zorder=3)
ax.set_xticks(x)
ax.set_xticklabels([f'{c[:4]}\n{h}h' for c, h in order], fontsize=8.5)
ax.set_ylabel('MAE  (µg/m³)')
ax.set_title('Adding the hour-of-day shape', loc='left', fontsize=12.5,
             fontweight='bold', pad=26)
ax.text(0, 1.06, 'The gain lands where the shape departs most from the daily mean — '
        'the afternoon and evening. At 9am it adds almost nothing.',
        transform=ax.transAxes, fontsize=9.5, color=MUTED)
ax.legend(frameon=False, fontsize=9.5, ncol=2)
ax.grid(axis='y', color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for s in ('top','right'): ax.spines[s].set_visible(False)
ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')
plt.tight_layout(); plt.show()

### What the shapes look like

Worth seeing directly, because the shape is what the "best time of day" advice is made of.

In [ ]:
fig, axes = city_grid(ncol=3, w=4.5, h=3.5, sharex=True, sharey=True)
for ax, city in zip(axes, CITIES):
    g = complete_days(hourly[hourly.city == city])
    if g.empty:
        ax.axis('off'); continue
    sh = diurnal_shape(g)
    for mo, lab, col in [(11, 'Nov (season)', '#b8541a'), (7, 'Jul (monsoon)', '#1baf7a')]:
        if mo in sh.index.get_level_values(0):
            v = sh.loc[mo].sort_index()
            ax.plot(v.index, v.values, linewidth=2, color=col, label=lab)
    ax.axhline(1.0, color='0.55', linewidth=1, linestyle='--')
    ax.axvspan(6, 21, color='#eda100', alpha=0.05)
    ax.set_title(city, loc='left', fontsize=11, fontweight='bold', pad=6)
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    for s in ('top','right'): ax.spines[s].set_visible(False)
    ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')
axes[0].legend(frameon=False, fontsize=8.5)
for ax in axes[-3:]: ax.set_xlabel('hour of day')
for i in range(0, len(axes), 3): axes[i].set_ylabel('ratio to the day’s mean')
fig.suptitle('Diurnal shape — ratio of each hour to that day’s mean',
             fontsize=13, fontweight='bold', x=0.06, ha='left', y=0.985)
fig.text(0.06, 0.94, 'Above the dashed line is dirtier than the daily average, below is '
         'cleaner. Shaded band is 6am–9pm, when people actually choose.',
         fontsize=9.5, color=MUTED, ha='left')
fig.tight_layout(rect=[0, 0, 1, 0.9]); plt.show()

### Does the advice hold up?

The test that matters for the feature in the mockups. Take the shape's three cleanest
daytime hours, then check where those hours *actually* ranked that day.

In [ ]:
bh = {c: best_hour_eval(hourly[hourly.city == c], TEST_YEAR)
      for c in CITIES if (hourly.city == c).any()}
bh = pd.DataFrame({k: v for k, v in bh.items() if v}).T.reindex(CITIES).dropna(how='all')
print('"BEST TIME OF DAY" — advice quality, season, test '
      f'{TEST_YEAR}')
print(bh.round(1).to_string())
print()
print('  mean_rank   : average true rank of a recommended hour (1 = cleanest)')
print('  random_rank : what you would get picking at random')
print('  ugm3_saved  : concentration avoided vs going out at an average time')

fig, ax = plt.subplots(figsize=(10, 4.4))
cities_ok = list(bh.index)
x = np.arange(len(cities_ok))
ax.bar(x, bh.ugm3_saved.values, color=[CITY_COLOR[c] for c in cities_ok], zorder=3)
for i, c in enumerate(cities_ok):
    ax.text(i, bh.ugm3_saved.iloc[i], f'  {bh.pct_in_best5.iloc[i]:.0f}% in best 5',
            ha='center', va='bottom', fontsize=9, color=MUTED)
ax.set_xticks(x); ax.set_xticklabels(cities_ok)
ax.set_ylabel('µg/m³ avoided')
ax.set_title('What following the advice is worth', loc='left', fontsize=12.5,
             fontweight='bold', pad=26)
ax.text(0, 1.06, 'Concentration avoided by going out at a recommended hour instead of an '
        'average one. Strongest exactly where it matters most.',
        transform=ax.transAxes, fontsize=9.5, color=MUTED)
ax.grid(axis='y', color=GRID, linewidth=0.8); ax.set_axisbelow(True)
ax.margins(y=0.18)
for s in ('top','right'): ax.spines[s].set_visible(False)
ax.spines['left'].set_color('0.7'); ax.spines['bottom'].set_color('0.7')
plt.tight_layout(); plt.show()

In Delhi the advice avoids **35 µg/m³** and lands in the day's true best five hours 72% of
the time — a material exposure reduction, in the city where exposure is worst.

In Hyderabad and Chennai it is close to a coin flip (51% and 48%) because their shapes are
flatter. **Those cities should suppress the recommendation when the predicted spread across
the day is small**, rather than state weak advice confidently — the same honesty rule the
rest of the app follows.

**So AQI can be forecast directly, and should be.** Skill is within about a point of PM2.5
in six of seven cities (Hyderabad is the exception at −10 points). The model does not care
that the target is a max over four pollutants — it is still a smooth regional series with a
seasonal cycle and short-lived departures from it.

What this rules out is the shortcut: forecasting PM2.5 and labelling the result AQI. On ~72%
of days that would be tracking the wrong pollutant.

Practically the nightly job should forecast **both** — PM2.5 because users ask for it by
name, and composite AQI because it is what the headline number shows.

---
## Choosing what to show, from how old the data is

OpenAQ's publication lag is not a constant — measured at 17 h, 28 h and 83 h on different
days, and individual stations go quiet for a week at a time. So the app cannot decide once
whether it is "showing a forecast"; it has to decide per station, per request, from the age
of that station's freshest reading.

This table is that decision, precomputed. For every combination of input age and horizon it
scores the forecast against climatology — the thing we fall back to, which needs no recent
data at all.

In [ ]:
from baselines import station_mode_table, max_useful_age

# station_mode_table measures the mode on the STATIONS and takes the median,
# which is how the mode is served. The city-average version promised skill at
# data ages where no individual monitor had any left.
modes = pd.concat([station_mode_table(station_pm25, c, TEST_YEAR).assign(city=c)
                   for c in CITIES], ignore_index=True)

print('DELHI — what to serve, by data age and horizon')
d = modes[modes.city == 'Delhi']
print(d.pivot(index='data_age_days', columns='horizon_days', values='mode').to_string())
print()
print('skill (% better than climatology):')
print(d.pivot(index='data_age_days', columns='horizon_days',
              values='skill_pct').round(1).to_string())
print()
for c in CITIES:
    m = modes[modes.city == c]
    if m.empty: continue
    print(f'  {c:<11} still a forecast at +1 day with input up to '
          f'{max_useful_age(m, 1)} days old')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

MODE_COLOR = {'forecast': '#1baf7a', 'outlook': '#eda100', 'seasonal_normal': '#c9d4d9'}
d = modes[modes.city == 'Delhi']
grid = d.pivot(index='data_age_days', columns='horizon_days', values='mode')
num = grid.replace({'forecast': 2, 'outlook': 1, 'seasonal_normal': 0}).astype(float)

ax = axes[0]
from matplotlib.colors import ListedColormap
ax.imshow(num.values, cmap=ListedColormap([MODE_COLOR['seasonal_normal'],
                                           MODE_COLOR['outlook'],
                                           MODE_COLOR['forecast']]),
          vmin=0, vmax=2, aspect='auto')
for i in range(num.shape[0]):
    for j in range(num.shape[1]):
        ax.text(j, i, grid.values[i, j].replace('seasonal_normal', 'seasonal'),
                ha='center', va='center', fontsize=8.5, color='#12171b')
ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels([f'+{c}d' for c in grid.columns])
ax.set_yticks(range(len(grid.index))); ax.set_yticklabels([f'{i}d old' for i in grid.index])
ax.set_xlabel('forecast for'); ax.set_ylabel('age of newest reading')
ax.set_title('Delhi — what the app should say', loc='left',
             fontsize=11.5, fontweight='bold', pad=8)
for sp in ('top','right','left','bottom'): ax.spines[sp].set_visible(False)
ax.tick_params(length=0)

ax = axes[1]
for c in CITIES:
    m = modes[(modes.city == c) & (modes.horizon_days == 1)].sort_values('data_age_days')
    if m.empty: continue
    ax.plot(m.data_age_days, m.skill_pct, marker='o', markersize=4,
            color=CITY_COLOR[c], label=c, linewidth=1.8)
ax.axhline(20, color='#1baf7a', linestyle='--', linewidth=1.2)
ax.axhline(5, color='#eda100', linestyle='--', linewidth=1.2)
ax.axhline(0, color='#a32219', linewidth=1.4)
ax.text(0.02, 21, 'forecast above here', transform=ax.get_yaxis_transform(),
        fontsize=8.5, color='#1baf7a')
ax.text(0.02, 6, 'outlook', transform=ax.get_yaxis_transform(),
        fontsize=8.5, color='#8a6108')
ax.set_xlabel('age of newest reading (days)')
ax.set_ylabel('% better than climatology')
ax.set_title('Skill decays with staleness, at +1 day', loc='left',
             fontsize=11.5, fontweight='bold', pad=8)
ax.legend(frameon=False, fontsize=8, ncol=4)
ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for sp in ('top','right'): ax.spines[sp].set_visible(False)

plt.tight_layout(); plt.show()

This is the table the serving layer reads. Given a station's freshest reading and the
horizon asked for, it returns a mode and a band width — so the app degrades honestly from
**forecast** to **outlook** to **seasonal normal** instead of presenting an average as a
prediction.

The thresholds are not arbitrary: **seasonal_normal** begins where measured skill falls
below 5%, which is inside the noise of the backtest itself, and 0% is exactly the seasonal
average computed the long way round.